#widgets

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("schemaBronze", "bronze")
dbutils.widgets.text("schemaSilver", "silver")
dbutils.widgets.text("catalogo", "project_dev")

In [0]:
schemaBronze = dbutils.widgets.get("schemaBronze")
schemaSilver = dbutils.widgets.get("schemaSilver")
catalogo = dbutils.widgets.get("catalogo")

# definir rutas
path_inicio = f"{catalogo}.{schemaBronze}"
path_target = f"{catalogo}.{schemaSilver}"

In [0]:
from pyspark.sql.functions import col, broadcast, when, lit
from pyspark.sql import functions as F

##customers

In [0]:

df_customers = spark.table( f"{catalogo}.{schemaBronze}.customers ") \
  .select( "customer_id", 
            "customer_city",
            "customer_state"
          )


##sellers

In [0]:
df_sellers = (
    spark.table( f" {catalogo}.{schemaBronze}.sellers")
    .select( 
            "seller_id", 
            "seller_city",
            "seller_state"
    )
)



## load tables

In [0]:
### guardar la tabla clean en el target
df_customers.write.mode("overwrite").insertInto(f"{path_target}.customers")

In [0]:
### guardar la tabla clean en el target
df_sellers.write.mode("overwrite").insertInto(f"{path_target}.sellers")

In [0]:
%sql
---- aplicar optimizer a la tabla target
OPTIMIZE ${catalogo}.${schemaSilver}.customers
ZORDER BY customer_id;

OPTIMIZE ${catalogo}.${schemaSilver}.sellers
ZORDER BY seller_id;

In [0]:
%sql
--- habilitar vacum dinamico
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

In [0]:
%sql
---- solo tener archivos inferiores  24 horas
VACUUM ${catalogo}.${schemaSilver}.customers RETAIN 24 HOURS;

VACUUM ${catalogo}.${schemaSilver}.sellers RETAIN 24 HOURS;

In [0]:
%sql
--- describir la historia de la tabla
DESCRIBE HISTORY ${catalogo}.${schemaSilver}.customers;